# Initial Model Evaluation Notebook

## Constants

Centralized configuration values used by the evaluation cells.

### Variables

In [ ]:
import utils

from ollama import Client

models = [
    "ministral-3:8b",
    "ministral-3:3b",
    "ministral-3:14b",
    "phi4:14b",
    "mistral-nemo:12b",
    "mistral:7b",
    "gemma4:e2b",
    "gemma4:e4b",
    "gemma4:26b",
    "llama3.1:8b",
    "deepseek-r1:8b",
    "qwen3.5:9b",
    "qwen3.6:35b-a3b",
]

OLLAMA_SERVER = "http://156.35.95.33:11434"
OLLAMA_OPTIONS = {}

SYSTEM_PROMPT = utils.SYSTEM_PROMPT
EVAL_QUERY = (
    "Music albums where the lead singer was born in the same country as the album's producer"
)
TEMPERATURE_CANDIDATES = [
    # 0.0125,
    0.025,
    0.05,
    0.1,
    0.15,
    # 0.2,
    # 0.25,
]
RUNS_PER_TEMPERATURE = 10
VARIABILITY_THRESHOLD = 20.0  # percent
THINKING = False
OUTPUT_DIR = "outputs/temperatures"

ollama_client = Client(OLLAMA_SERVER)

In [6]:
all_results = {}
all_recommended = {}

for model_name in models:
    results = utils.evaluate_temperature_candidates(
        ollama_client=ollama_client,
        model_name=model_name,
        query=EVAL_QUERY,
        system_prompt=SYSTEM_PROMPT,
        base_options=OLLAMA_OPTIONS,
        temperature_candidates=TEMPERATURE_CANDIDATES,
        runs_per_temperature=RUNS_PER_TEMPERATURE,
        variability_threshold=VARIABILITY_THRESHOLD,
        thinking=THINKING,
    )

    report = utils.build_temperature_evaluation_report(
        model_name=model_name,
        query=EVAL_QUERY,
        threshold=VARIABILITY_THRESHOLD,
        results=results,
    )

    print(report)
    print("\n" + "=" * 80 + "\n")

    saved_path = utils.save_temperature_report(OUTPUT_DIR, model_name, report)
    print(f"Saved report: {saved_path}\n")

    all_results[model_name] = results
    all_recommended[model_name] = utils.select_closest_below_threshold(
        results, VARIABILITY_THRESHOLD
    )

Model: ministral-3:14b
Query: Music albums where the lead singer was born in the same country as the album's producer
Threshold: variability <= 20.0%

Results by temperature:
- T=0.025: variability=0.0% | unique=10.0% | under_threshold=OK
- T=0.05: variability=0.0% | unique=10.0% | under_threshold=OK
- T=0.1: variability=19.51% | unique=30.0% | under_threshold=OK
- T=0.15: variability=25.9% | unique=50.0% | under_threshold=NO

Recommended temperature (closest variability below threshold): 0.1 with variability=19.51%


Saved report: outputs\temperatures\ministral-3_14b.txt

